In [ ]:
# Example 1 — the core runtime-init surface: Accelerator(cpu=True) prepares a
# tiny model + optimizer + dataloader, then runs ONE real backward/step.
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from accelerate import Accelerator
from accelerate.utils import set_seed

set_seed(0)
acc = Accelerator(cpu=True)
print(f"device={acc.device} num_processes={acc.num_processes} mixed_precision={acc.mixed_precision}")

model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
X, Y = torch.randn(32, 4), torch.randint(0, 2, (32,))
loader = DataLoader(TensorDataset(X, Y), batch_size=8)
model, opt, loader = acc.prepare(model, opt, loader)

w0 = model.weight.detach().clone()
for xb, yb in loader:
    opt.zero_grad()
    loss = nn.functional.cross_entropy(model(xb), yb)
    acc.backward(loss)
    opt.step()
    break
assert not torch.equal(w0, model.weight.detach()), "one training step must update weights"
print(f"one step done, loss={loss.item():.4f}, weights updated")

In [ ]:
# Example 2 — collective ops (single-process no-ops here, but exercise the real
# code path) + a save_state -> load_state checkpoint round-trip.
import os, tempfile, torch, torch.nn as nn
from accelerate import Accelerator

acc = Accelerator(cpu=True)
t = torch.arange(4, dtype=torch.float32)
assert torch.equal(acc.gather(t), t), "single-proc gather is identity"
assert acc.reduce(t.clone(), reduction="sum").sum().item() == t.sum().item()

model = nn.Linear(8, 8)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
model, opt = acc.prepare(model, opt)
ckpt = tempfile.mkdtemp()
acc.save_state(ckpt)
ref = acc.unwrap_model(model).weight.detach().clone()
with torch.no_grad():
    acc.unwrap_model(model).weight.zero_()
acc.load_state(ckpt)
restored = acc.unwrap_model(model).weight.detach()
assert torch.allclose(ref, restored), "load_state must restore saved weights"
print(f"gather/reduce OK; checkpoint restored {restored.numel()} params from {len(os.listdir(ckpt))} files")

In [ ]:
# Example 3 — big-model-inference: init_empty_weights builds a model on the
# `meta` device (zero RAM), a signature accelerate feature for large models.
import torch, torch.nn as nn
from accelerate import init_empty_weights
from accelerate.utils import get_max_memory, compute_module_sizes, is_mps_available

with init_empty_weights():
    model = nn.Sequential(nn.Linear(2048, 2048), nn.ReLU(), nn.Linear(2048, 512))
assert all(p.device.type == "meta" for p in model.parameters()), "params must be on meta device"
n_params = sum(p.numel() for p in model.parameters())
print(f"built {n_params:,}-param model on meta device (no allocation)")

sizes = compute_module_sizes(model)
assert sizes[""] > 0, "root module must report a positive byte size"
print(f"compute_module_sizes root bytes={sizes['']:,}")
mm = get_max_memory()
_integrated = [
    index
    for index in range(torch.cuda.device_count())
    if getattr(torch.cuda.get_device_properties(index), "is_integrated", False)
]
assert mm, "get_max_memory must report at least one memory budget"
assert all(isinstance(value, int) and value > 0 for value in mm.values()), (
    f"every budget must be a positive byte count, got {mm}"
)
if is_mps_available():
    # Apple-silicon MPS shares host RAM: get_max_memory reports an mps budget and no separate cpu one.
    assert "mps" in mm and "cpu" not in mm, f"MPS host: expected an mps budget and no cpu budget, keys={list(mm)}"
else:
    assert ("cpu" in mm) == (not _integrated), (
        f"get_max_memory exposes a separate cpu budget exactly when no visible CUDA device is "
        f"integrated; integrated devices={_integrated}, keys={list(mm)}"
    )
print(f"get_max_memory keys={list(mm)[:3]} (integrated CUDA devices: {_integrated})")

In [ ]:
# Example 4 — determinism invariant (set_seed reproducibility) + the
# find_executable_batch_size auto-retry decorator + PartialState introspection.
import torch
from accelerate.utils import set_seed, find_executable_batch_size
from accelerate import PartialState

set_seed(1234); a = torch.randn(6)
set_seed(1234); b = torch.randn(6)
assert torch.equal(a, b), "set_seed must make RNG reproducible"

@find_executable_batch_size(starting_batch_size=16)
def forward(batch_size):
    return (torch.randn(batch_size, 64) @ torch.randn(64, 64)).shape

out_shape = forward()
print(f"determinism OK; find_executable_batch_size produced out_shape={tuple(out_shape)}")

state = PartialState()
print(f"PartialState: process {state.process_index}/{state.num_processes} on {state.device}, distributed={state.distributed_type}")

In [ ]:
# Example 5 — GPU. Accelerator device selection + a device-resident forward + matmul.
# Cells above deliberately ran Accelerator(cpu=True), which pins the process-wide
# AcceleratorState/PartialState singleton to CPU — reset it first so THIS cell
# exercises real GPU device selection (the idiom accelerate's own tests use).
import torch, torch.nn as nn
from accelerate import Accelerator
from accelerate.state import AcceleratorState

assert torch.cuda.is_available(), (
    f"requires-gpu cell ran but torch.cuda.is_available() is False — torch {torch.__version__}, "
    f"built against CUDA {torch.version.cuda}, device_count={torch.cuda.device_count()}"
)

AcceleratorState._reset_state(reset_partial_state=True)
acc = Accelerator()
assert acc.device.type == "cuda", f"expected cuda, got {acc.device}"
model = acc.prepare(nn.Linear(128, 128))
x = torch.randn(64, 128, device=acc.device)
y = model(x)
assert y.is_cuda and tuple(y.shape) == (64, 128), "forward must stay on-device"
z = x @ x.t()
torch.cuda.synchronize()
print(f"GPU run on {acc.device}: prepared-model output {tuple(y.shape)}, matmul {tuple(z.shape)}")